# 02 - Data Processing and Cleaning
Loads the raw CSVs, merges them into one dataframe, cleans it, engineers pre-race features, and splits by season into train / validation / test.

## 1. Load the CSV files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp /content/drive/MyDrive/results.csv /content/drive/MyDrive/quali.csv /content/drive/MyDrive/weather.csv .

In [ ]:
import pandas as pd
import numpy as np

results = pd.read_csv("results.csv")
quali = pd.read_csv("quali.csv")
weather = pd.read_csv("weather.csv")

print("results:", results.shape, "| quali:", quali.shape, "| weather:", weather.shape)

## 2. Merge into one dataframe

In [ ]:
df = (results
      .merge(quali, on=["season", "round", "driverId"], how="left")
      .merge(weather, on=["season", "round"], how="left"))

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "finish_pos"]).reset_index(drop=True)

print("Rows after merge:", len(df), "| rows in results:", len(results))   # must match
print("Duplicate driver-race rows:", df.duplicated(["season", "round", "driverId"]).sum())  # must be 0
print()
print(df.isna().sum())

## 3. Clean
- Pit-lane starts are stored as grid 0, so they are treated as last on the grid.
- Missing qualifying positions fall back to the grid position.
- Weather stays empty where it is missing (2018-2020, plus any races not yet fetched). XGBoost handles missing values, so we do not fill it.

In [ ]:
df["grid"] = df["grid"].replace(0, 20)
df["quali_pos"] = df["quali_pos"].fillna(df["grid"])

# DNF flag from the status text (use int so rolling averages work)
df["dnf"] = (~(df["status"].eq("Finished") | df["status"].str.startswith("+"))).astype(int)

# Targets
df["podium"] = (df["finish_pos"] <= 3).astype(int)
df["won"] = (df["finish_pos"] == 1).astype(int)

## 4. Sanity checks

In [ ]:
per_race = df.groupby(["season", "round"]).agg(drivers=("driverId", "count"),
                                               podium=("podium", "sum"))

print("Podium finishers per race (almost all should be 3):")
print(per_race["podium"].value_counts())
print()
print("Drivers per race:")
print(per_race["drivers"].describe().round(1))
print()
print("Overall podium rate:", round(df["podium"].mean(), 3), "(expect about 0.15)")

odd = per_race[per_race["podium"] != 3]
if len(odd):
    print("\nRaces without exactly 3 podium finishers - check these:")
    print(odd)

## 5. Feature engineering
Every rolling feature uses `shift(1)`, so a race only sees information from earlier races. This prevents leakage.

In [ ]:
def add_features(df, n=5):
    df = df.sort_values(["date", "finish_pos"]).reset_index(drop=True).copy()

    # Driver form
    g = df.groupby("driverId")
    df["drv_form_finish"] = g["finish_pos"].transform(lambda s: s.shift(1).rolling(n, min_periods=1).mean())
    df["drv_form_points"] = g["points"].transform(lambda s: s.shift(1).rolling(n, min_periods=1).mean())
    df["drv_dnf_rate"] = g["dnf"].transform(lambda s: s.shift(1).expanding().mean())
    df["drv_win_rate"] = g["won"].transform(lambda s: s.shift(1).expanding().mean())

    # Driver history at this circuit
    df["drv_circuit_avg_finish"] = (df.groupby(["driverId", "circuitId"])["finish_pos"]
                                      .transform(lambda s: s.shift(1).expanding().mean()))

    # Constructor strength: team points per race, rolling over the last n races
    team = (df.groupby(["constructorId", "date"], as_index=False)["points"].sum()
              .rename(columns={"points": "team_points"})
              .sort_values("date"))
    team["team_form_points"] = (team.groupby("constructorId")["team_points"]
                                    .transform(lambda s: s.shift(1).rolling(n, min_periods=1).mean()))
    df = df.merge(team[["constructorId", "date", "team_form_points"]],
                  on=["constructorId", "date"], how="left")

    # Neutral defaults for a driver's or team's first races
    df["drv_form_finish"] = df["drv_form_finish"].fillna(10)
    df["drv_circuit_avg_finish"] = df["drv_circuit_avg_finish"].fillna(10)
    for c in ["drv_form_points", "drv_dnf_rate", "drv_win_rate", "team_form_points"]:
        df[c] = df[c].fillna(0)
    return df

data = add_features(df)
print(data.shape)
data[["season", "round", "driverId", "grid", "drv_form_finish", "drv_dnf_rate",
      "team_form_points", "podium"]].head(10)

## 6. Split by season

In [ ]:
train = data[data["season"] <= 2023]
val   = data[data["season"] == 2024]
test  = data[data["season"] >= 2025]

for name, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} rows={len(d):5d} races={d.groupby(['season','round']).ngroups:3d} podium rate={d['podium'].mean():.3f}")

## 7. Save the final dataset

In [ ]:
data.to_csv("f1_final.csv", index=False)
train.to_csv("f1_train.csv", index=False)
val.to_csv("f1_val.csv", index=False)
test.to_csv("f1_test.csv", index=False)

!cp f1_final.csv f1_train.csv f1_val.csv f1_test.csv /content/drive/MyDrive/
print("Saved to Google Drive.")